# Phase 6 — Association Rule Mining
**AgroSense AI Pipeline**

Algorithm: Apriori (via mlxtend)

Goals:
- Discretise continuous features into categorical bins
- Mine frequent itemsets (min_support=0.05)
- Generate strong rules (min_confidence=0.60)
- Visualise rules by lift and confidence
- Extract crop-specific planting rules for the app

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,
                     'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3,'figure.dpi':120})

with open('processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

df = data['df_clean'].copy()
print(f'Loaded {len(df)} records.')

## 6.1  Feature Discretisation

In [ ]:
df_disc = pd.DataFrame()
df_disc['N_cat']    = pd.cut(df['n'],           bins=[0,40,80,200],   labels=['N_low','N_med','N_high'])
df_disc['P_cat']    = pd.cut(df['p'],           bins=[0,40,80,145],   labels=['P_low','P_med','P_high'])
df_disc['K_cat']    = pd.cut(df['k'],           bins=[0,40,80,205],   labels=['K_low','K_med','K_high'])
df_disc['Temp_cat'] = pd.cut(df['temperature'], bins=[0,20,30,55],    labels=['Temp_low','Temp_med','Temp_high'])
df_disc['Hum_cat']  = pd.cut(df['humidity'],    bins=[0,50,80,100],   labels=['Hum_low','Hum_med','Hum_high'])
df_disc['pH_cat']   = pd.cut(df['ph'],          bins=[0,5.5,7.5,14],  labels=['pH_acid','pH_neutral','pH_alkaline'])
df_disc['Rain_cat'] = pd.cut(df['rainfall'],    bins=[0,60,150,400],  labels=['Rain_low','Rain_med','Rain_high'])
df_disc['Crop']     = df['label']

print('Discretised dataset:')
print(df_disc.head())
print(f'\nMissing after binning: {df_disc.isnull().sum().sum()}')
df_disc.dropna(inplace=True)
print(f'Records after dropping nulls: {len(df_disc)}')

## 6.2  Build Transaction Matrix

In [ ]:
ARM_AVAILABLE = False
try:
    from mlxtend.frequent_patterns import apriori, association_rules
    from mlxtend.preprocessing import TransactionEncoder
    ARM_AVAILABLE = True
    print('mlxtend available — running full Apriori.')
except ImportError:
    print('mlxtend not installed. Run: pip install mlxtend')
    print('Showing code for reference.')

In [ ]:
if ARM_AVAILABLE:
    transactions = []
    for _, row in df_disc.iterrows():
        items = [str(row[c]) for c in df_disc.columns]
        transactions.append(items)
    
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    te_df = pd.DataFrame(te_array, columns=te.columns_)
    
    print(f'Transaction matrix shape: {te_df.shape}')
    print(f'Total items (columns): {te_df.shape[1]}')

## 6.3  Mine Frequent Itemsets + Rules

In [ ]:
if ARM_AVAILABLE:
    freq_items = apriori(te_df, min_support=0.04, use_colnames=True, max_len=4)
    freq_items['length'] = freq_items['itemsets'].apply(len)
    
    print(f'Frequent itemsets found: {len(freq_items)}')
    print(freq_items.groupby('length').size().rename('count').to_frame())
    
    rules = association_rules(freq_items, metric='confidence', min_threshold=0.55)
    rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
    
    rules_sorted = rules.sort_values('lift', ascending=False)
    print(f'\nTotal rules: {len(rules_sorted)}')
    print(f'Rules with lift > 3: {(rules_sorted["lift"] > 3).sum()}')
    
    rules_sorted[['antecedents','consequents','support','confidence','lift']].head(20).round(3)

## 6.4  Visualise Rules

In [ ]:
if ARM_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Support vs Confidence scatter
    scatter = axes[0].scatter(rules_sorted['support'], rules_sorted['confidence'],
                               c=rules_sorted['lift'], cmap='Greens',
                               alpha=0.6, s=20)
    plt.colorbar(scatter, ax=axes[0], label='Lift')
    axes[0].set_xlabel('Support')
    axes[0].set_ylabel('Confidence')
    axes[0].set_title('Support vs Confidence', fontweight='bold')
    
    # Lift distribution
    axes[1].hist(rules_sorted['lift'], bins=40, color='#2E7D32', alpha=0.8)
    axes[1].axvline(1.0, color='#C62828', linewidth=1.5, linestyle='--', label='Lift=1 (random)')
    axes[1].set_xlabel('Lift')
    axes[1].set_ylabel('Number of Rules')
    axes[1].set_title('Lift Distribution', fontweight='bold')
    axes[1].legend()
    
    # Top rules by lift
    top10 = rules_sorted.head(10)
    axes[2].barh(range(10), top10['lift'].values[::-1], color='#2E7D32', alpha=0.8)
    axes[2].set_yticks(range(10))
    axes[2].set_yticklabels([f"{row['consequents'][:15]}" for _, row in top10.iloc[::-1].iterrows()],
                              fontsize=8)
    axes[2].set_xlabel('Lift')
    axes[2].set_title('Top 10 Rules by Lift', fontweight='bold')
    
    plt.suptitle('Association Rules Analysis', fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('outputs/phase6_arm_analysis.png', bbox_inches='tight')
    plt.show()

else:
    # Show static example rules
    example = pd.DataFrame({
        'antecedents': ['N_high, Rain_high','N_low, K_low, Hum_high',
                        'Temp_high, Rain_low, Hum_low',
                        'P_high, K_high, pH_neutral',
                        'Rain_low, Temp_high, N_low'],
        'consequents': ['rice','chickpea','cotton','coconut','groundnut'],
        'support':     [0.091, 0.073, 0.064, 0.055, 0.052],
        'confidence':  [0.913, 0.854, 0.831, 0.792, 0.778],
        'lift':        [4.21,  3.94,  3.64,  3.21,  3.08],
    })
    print('Example ARM rules (static):')
    print(example.to_string(index=False))

## 6.5  Crop-Specific Rule Extraction

In [ ]:
if ARM_AVAILABLE:
    # Extract rules where consequent is a crop label
    crop_rules = rules_sorted[
        rules_sorted['consequents'].str.lower().apply(
            lambda x: any(c in x for c in data['class_names'])
        )
    ].copy()
    
    print(f'Crop-specific rules: {len(crop_rules)}')
    print('\nTop 15 crop planting rules:')
    print(crop_rules[['antecedents','consequents','confidence','lift']].head(15).to_string(index=False))
    
    # Save to processed data
    data['arm_rules'] = crop_rules[['antecedents','consequents','support','confidence','lift']].head(30)
    with open('processed_data.pkl', 'wb') as f:
        pickle.dump(data, f)
    print('\nARM rules saved to processed_data.pkl')